# Animal Stats Analysis

Per-animal evolution of summary statistics, cross-stat correlations, and correlation with session index.

Works on real data (loaded via `build_feature_matrix`) and optionally compares with matched synthetic animals.

**Sections:**
1. Load real data
2. Per-animal stat trajectories
3. Cross-stat correlation matrix
4. Stat vs session index
5. Summary across animals

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

from Data.structures import load_animal, load_experiment
from Analysis.session_features import build_feature_matrix, build_feature_matrix_multi
from Analysis.summary_stats import list_available_stats, FEATURE_MATRIX_STATS


## Config

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH = '/Users/Serkan/Desktop/pro/PhD/main/data/Head_Fixed_Behavior/Processed'

# ── Which animals to load (None = all in DATA_PATH) ──────────────────────────
ANIMAL_IDS = None   # e.g. ['SS05', 'SS06'] or None

# ── Stage filter ─────────────────────────────────────────────────────────────
STAGE = 'Full_Task_Cont'   # None = all stages

# ── Stats for trajectory/correlation analysis ─────────────────────────────────
# These are the stats that will appear in plots.
# subset of FEATURE_MATRIX_STATS or list_available_stats()
TRAJ_STATS = [
    'accuracy', 'pse', 'slope', 'recency', 'win_stay', 'lose_shift',
    'stimulus_sensitivity', 'choice_entropy', 'perseveration',
    'hard_accuracy', 'easy_accuracy',
]

# ── Minimum sessions to include an animal ─────────────────────────────────────
MIN_SESSIONS = 3


## 1. Load Data

In [ ]:
experiment = load_experiment(DATA_PATH, animal_ids=ANIMAL_IDS)
animals    = list(experiment.animals.values())
print(f"Loaded {len(animals)} animals:")
for a in animals:
    print(f"  {a.animal_id}: {a.n_sessions} sessions")


In [ ]:
# Build feature matrix (one row per session, all stats + metadata)
df_all = build_feature_matrix_multi(
    animals,
    stage=STAGE,
    compute_deltas=True,
)

print(f"Feature matrix: {len(df_all)} sessions × {len(df_all.columns)} features")
print(f"Animals: {df_all['animal_id'].unique()}")
print(f"Sessions per animal:\n{df_all.groupby('animal_id')['session_idx'].count()}")

# Keep only animals with enough sessions
counts   = df_all.groupby('animal_id')['session_idx'].count()
keep_ids = counts[counts >= MIN_SESSIONS].index.tolist()
df       = df_all[df_all['animal_id'].isin(keep_ids)].copy()
print(f"\nAfter filtering (>={MIN_SESSIONS} sessions): {len(df)} sessions, {len(keep_ids)} animals")


In [ ]:
# Confirm which TRAJ_STATS are actually in the DataFrame
available = [s for s in TRAJ_STATS if s in df.columns]
missing   = [s for s in TRAJ_STATS if s not in df.columns]
if missing:
    print(f"Warning: stats not found in DataFrame, skipping: {missing}")
TRAJ_STATS = available
print(f"Using {len(TRAJ_STATS)} stats: {TRAJ_STATS}")


## 2. Per-Animal Stat Trajectories

One subplot per stat. Each animal is a separate line. Session index on x-axis.

In [ ]:
animal_ids = df['animal_id'].unique()
n_stats    = len(TRAJ_STATS)
n_cols     = 3
n_rows     = int(np.ceil(n_stats / n_cols))

cmap_a = plt.cm.tab10
colors = {aid: cmap_a(i % 10) for i, aid in enumerate(animal_ids)}

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5*n_cols, 3.8*n_rows), sharex=False)
axes_flat  = axes.flat if hasattr(axes, 'flat') else [axes]
fig.suptitle('Summary Stat Trajectories per Animal', fontsize=14, fontweight='bold')

for ax, sname in zip(axes_flat, TRAJ_STATS):
    for aid in animal_ids:
        sub = df[df['animal_id'] == aid].sort_values('session_idx')
        x   = sub['session_idx'].values
        y   = sub[sname].values
        ax.plot(x, y, 'o-', color=colors[aid], lw=1.5, ms=4,
                alpha=0.85, label=aid)
    ax.set(xlabel='Session index', ylabel=sname, title=sname)
    ax.axhline(0, color='k', lw=0.4, ls='--', alpha=0.3)

# Legend in last used axis
handles = [plt.Line2D([0], [0], color=colors[aid], lw=2, label=aid)
           for aid in animal_ids]
axes_flat[-1 % (n_rows*n_cols)].legend(handles=handles, fontsize=8, title='Animal')

# Hide unused subplots
for ax in list(axes_flat)[len(TRAJ_STATS):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


## 3. Cross-Stat Correlation Matrix

Are stats redundant? Spearman correlations pooled across all sessions and animals.

In [ ]:
stat_data = df[TRAJ_STATS].values   # (n_sessions, n_stats)

n = len(TRAJ_STATS)
corr_mat  = np.full((n, n), np.nan)
pval_mat  = np.full((n, n), np.nan)

for i in range(n):
    for j in range(n):
        xi, xj = stat_data[:, i], stat_data[:, j]
        valid   = ~(np.isnan(xi) | np.isnan(xj))
        if valid.sum() >= 10:
            rho, pval = spearmanr(xi[valid], xj[valid])
            corr_mat[i, j] = rho
            pval_mat[i, j] = pval

fig, ax = plt.subplots(figsize=(max(8, n*0.6), max(7, n*0.55)))
im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(n)); ax.set_xticklabels(TRAJ_STATS, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(TRAJ_STATS, fontsize=8)

# Annotate significant cells (p < 0.05 after Bonferroni)
thresh = 0.05 / (n*n)
for i in range(n):
    for j in range(n):
        if not np.isnan(corr_mat[i, j]):
            txt_col = 'white' if abs(corr_mat[i,j]) > 0.6 else 'black'
            marker  = '-' if pval_mat[i,j] > thresh else f'{corr_mat[i,j]:.2f}'
            ax.text(j, i, marker, ha='center', va='center', fontsize=6, color=txt_col)

plt.colorbar(im, ax=ax, label='Spearman ρ', shrink=0.8)
ax.set_title('Cross-Stat Correlation Matrix (pooled across animals and sessions)', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Per-animal correlation matrices (to check consistency across animals)
fig, axes = plt.subplots(1, len(animal_ids),
                         figsize=(4*len(animal_ids), 4), sharey=True)
if len(animal_ids) == 1:
    axes = [axes]

for ax, aid in zip(axes, animal_ids):
    sub      = df[df['animal_id'] == aid][TRAJ_STATS].values
    cm_local = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            xi, xj = sub[:, i], sub[:, j]
            valid   = ~(np.isnan(xi) | np.isnan(xj))
            if valid.sum() >= 5:
                rho, _ = spearmanr(xi[valid], xj[valid])
                cm_local[i, j] = rho
    im = ax.imshow(cm_local, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_title(aid, fontsize=9)
    ax.set_xticks(range(n)); ax.set_xticklabels(TRAJ_STATS, rotation=90, fontsize=6)
    if ax is axes[0]:
        ax.set_yticks(range(n)); ax.set_yticklabels(TRAJ_STATS, fontsize=6)

fig.colorbar(im, ax=axes[-1], label='ρ', shrink=0.8)
fig.suptitle('Cross-Stat Correlations — per Animal', fontsize=11)
plt.tight_layout()
plt.show()


## 4. Stat vs Session Index

Which stats change over learning? Spearman correlation of each stat with session index, computed within each animal then summarised.

In [ ]:
# Within-animal Spearman correlation of each stat with session index
per_animal_corrs = np.full((len(animal_ids), len(TRAJ_STATS)), np.nan)

for a_idx, aid in enumerate(animal_ids):
    sub = df[df['animal_id'] == aid].sort_values('session_idx')
    sess = sub['session_idx'].values.astype(float)
    for s_idx, sname in enumerate(TRAJ_STATS):
        y     = sub[sname].values.astype(float)
        valid = ~np.isnan(y)
        if valid.sum() >= 5:
            rho, _ = spearmanr(sess[valid], y[valid])
            per_animal_corrs[a_idx, s_idx] = rho

median_corr = np.nanmedian(per_animal_corrs, axis=0)
order       = np.argsort(median_corr)   # sorted low to high

fig, ax = plt.subplots(figsize=(max(10, len(TRAJ_STATS)*0.7), 5))
names_ordered = [TRAJ_STATS[i] for i in order]
meds_ordered  = median_corr[order]

bar_colors = ['#d32f2f' if v < 0 else '#1976d2' for v in meds_ordered]
ax.bar(range(len(order)), meds_ordered, color=bar_colors, alpha=0.8, zorder=2)

# Individual animal dots
for a_idx in range(len(animal_ids)):
    ax.scatter(range(len(order)), per_animal_corrs[a_idx, order],
               alpha=0.5, s=20, c='grey', zorder=3)

ax.set_xticks(range(len(order)))
ax.set_xticklabels(names_ordered, rotation=45, ha='right', fontsize=9)
ax.axhline(0,    color='k',    lw=0.8)
ax.axhline(0.3,  color='grey', lw=0.6, ls='--', alpha=0.5)
ax.axhline(-0.3, color='grey', lw=0.6, ls='--', alpha=0.5)
ax.set(ylabel='Spearman ρ (stat vs session index)',
       title='Which Stats Track Learning?'
             '(bars = median across animals; dots = individual animals; '
             'negative = decreases with learning)')
plt.tight_layout()
plt.show()


In [ ]:
# Trajectory of the top tracking stats over sessions
# Normalise each stat to [0,1] within animal to enable overlay
TOP_N = 5
abs_order = np.argsort(np.abs(median_corr))[::-1]
top_stats  = [TRAJ_STATS[i] for i in abs_order[:TOP_N]]
print(f"Top {TOP_N} stats by |corr with session index|: {top_stats}")

fig, ax = plt.subplots(figsize=(10, 5))
cmap_s = plt.cm.Set1

for s_idx, sname in enumerate(top_stats):
    # Normalise and average across animals
    traces = []
    for aid in animal_ids:
        sub = df[df['animal_id'] == aid].sort_values('session_idx')
        y   = sub[sname].values.astype(float)
        lo, hi = np.nanmin(y), np.nanmax(y)
        if hi > lo:
            traces.append((sub['session_idx'].values, (y - lo) / (hi - lo)))

    if not traces:
        continue

    # Interpolate to common session grid and plot mean ± std
    max_sess = max(t[0][-1] for t in traces)
    sess_grid = np.arange(0, int(max_sess)+1)
    interped  = np.full((len(traces), len(sess_grid)), np.nan)
    for k, (x, y) in enumerate(traces):
        for xi, yi in zip(x.astype(int), y):
            if not np.isnan(yi):
                interped[k, xi] = yi

    mu  = np.nanmean(interped, axis=0)
    sem = np.nanstd(interped, axis=0) / np.sqrt(np.sum(~np.isnan(interped), axis=0).clip(1))
    valid = ~np.isnan(mu)

    c = cmap_s(s_idx / len(top_stats))
    ax.plot(sess_grid[valid], mu[valid], '-', color=c, lw=2, label=sname)
    ax.fill_between(sess_grid[valid], (mu-sem)[valid], (mu+sem)[valid],
                    color=c, alpha=0.15)

ax.set(xlabel='Session index', ylabel='Normalised stat (0–1 per animal)',
       title=f'Top {TOP_N} Learning-Tracking Stats (mean ± SEM across animals)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 5. Summary Across Animals

Early (first third) vs late (last third) sessions for each stat.

In [ ]:
# For each animal, split sessions into early / late thirds and compute mean stat
early_vals = {s: [] for s in TRAJ_STATS}
late_vals  = {s: [] for s in TRAJ_STATS}

for aid in animal_ids:
    sub   = df[df['animal_id'] == aid].sort_values('session_idx')
    n_s   = len(sub)
    third = max(1, n_s // 3)
    early = sub.iloc[:third]
    late  = sub.iloc[-third:]
    for sname in TRAJ_STATS:
        early_vals[sname].append(np.nanmean(early[sname].values))
        late_vals[sname].append(np.nanmean(late[sname].values))

# Plot: paired scatter / bar for each stat
n_st  = len(TRAJ_STATS)
n_c   = 3
n_r   = int(np.ceil(n_st / n_c))

fig, axes = plt.subplots(n_r, n_c, figsize=(5*n_c, 3.5*n_r))
axes_flat  = axes.flat if hasattr(axes, 'flat') else [axes]
fig.suptitle('Early (first third) vs Late (last third) sessions',
             fontsize=13, fontweight='bold')

for ax, sname in zip(axes_flat, TRAJ_STATS):
    ev = np.array(early_vals[sname])
    lv = np.array(late_vals[sname])

    for i in range(len(animal_ids)):
        c = colors[animal_ids[i]]
        ax.plot([0, 1], [ev[i], lv[i]], 'o-', color=c, lw=1.5, ms=6, alpha=0.7)

    # Mean line
    ax.plot([0, 1], [np.nanmean(ev), np.nanmean(lv)],
            's--', color='black', lw=2.5, ms=8, zorder=5, label='mean')

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Early', 'Late'])
    ax.set(ylabel=sname, title=sname)

for ax in list(axes_flat)[n_st:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# Distribution of n_trials per session (sanity check)
fig, ax = plt.subplots(figsize=(8, 4))
for aid in animal_ids:
    sub = df[df['animal_id'] == aid].sort_values('session_idx')
    ax.plot(sub['session_idx'], sub['n_trials_valid'], 'o-',
            color=colors[aid], lw=1.5, ms=4, label=aid)

ax.set(xlabel='Session index', ylabel='Valid trials per session',
       title='Trial Counts over Sessions')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("\nSession counts by animal:")
print(df.groupby('animal_id')[['n_trials_total','n_trials_valid','abort_rate']].describe().round(2))
